In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install ffmpeg-python 
%pip install librosa soundfile
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --no-cache-dir --force-reinstall "numpy==1.26.4" "pandas==2.2.2" "opencv-python-headless==4.8.1.78"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 198.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 166.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 156.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata━━━━━━━━━━━━━━━━━━ 0/7 [pytz]
    Found existing installation: tzdata 2025.20m 0/7 [pytz]
    Uninstalling tzdata-2025.2:━━━━━━━━━━━━━ 0/7 [pytz]
      Successfully uninstalled tzdata-2025.2 0/7 [pytz]
  Attempting uninstall: six━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [tzdata]
    Found existing installation: six 1.17.0━━━━━━━━━━━━━━━━━━━ 1/7 [tzdata]
    Uninstalling six-1.17.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [tzdata]
      Successfully uninstalled six-1.17.0━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [six]
  Attempting uninstall: numpy━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [six]
    Fo

In [4]:
import os, math, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd

In [5]:
import librosa, librosa.display
import soundfile as sf
import cv2
from ultralytics import YOLO  # YOLOv8

In [17]:
# Pfade
VIDEO_PATH = Path("videos/test_video.mp4")
WORKDIR = Path("work_cat_poc"); WORKDIR.mkdir(exist_ok=True)
AUDIO_WAV = WORKDIR/"audio_16k_mono.wav"
FRAMES_DIR = WORKDIR/"frames"; FRAMES_DIR.mkdir(exist_ok=True)
SEGMENTS_CSV = WORKDIR/"segments.csv"
CLIPS_DIR = WORKDIR / "clips"; CLIPS_DIR.mkdir(exist_ok=True)

assert VIDEO_PATH.exists(), f"Video nicht gefunden: {VIDEO_PATH.resolve()}"

# Audio Extrahieren

In [7]:
import ffmpeg

_ = (
    ffmpeg
    .input(str(VIDEO_PATH))
    .output(str(AUDIO_WAV), ac=1, ar=16000, vn=None, loglevel="error")
    .overwrite_output()
    .run()
)
print("Audio extrahiert:", AUDIO_WAV.exists(), AUDIO_WAV)

Audio extrahiert: True work_cat_poc/audio_16k_mono.wav


# „Miau“-Kandidaten aus Audio (einfach & robust)

Heuristik: Nicht-Stille + dominante mittlere/hohe Frequenzen → grobe Miau-Kandidaten.

In [8]:
y, sr = librosa.load(AUDIO_WAV, sr=16000, mono=True)

# 3a) Nicht-Stille finden
non_silent_intervals = librosa.effects.split(y, top_db=25)  # toleranter Schwellwert
intervals = []

# 3b) Frequenz-Heuristik pro Intervall (Spektral-Zentroid)
for start, end in non_silent_intervals:
    seg = y[start:end]
    if len(seg) < 0.15*sr:  # <150ms verwerfen
        continue
    S = np.abs(librosa.stft(seg, n_fft=1024, hop_length=256))
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr).mean()
    # grob: "miau" häufig zwischen ~800–5000 Hz prominent
    if 800 <= centroid <= 5000:
        t0 = start/sr; t1 = end/sr
        energy = float(np.sqrt(np.mean(seg**2)))
        intervals.append({"t_start": t0, "t_end": t1, "audio_score": energy, "centroid": centroid})

audio_df = pd.DataFrame(intervals)
print("Audio-Kandidaten:", len(audio_df))
audio_df.head()

Audio-Kandidaten: 536


,t_start,t_end,audio_score,centroid
0,1.248,2.208,0.038460,2103.753811
1,2.304,5.120,0.071013,1639.279980
2,5.152,7.744,0.063584,1627.847077
3,8.256,8.864,0.019622,862.397881
4,9.152,9.344,0.023320,814.732260


# „Katze im Bild“: YOLOv8-Detektion (auf sparsamen Keyframes)

Wir sampeln z. B. alle 0,25 s einen Frame und prüfen, ob Klasse cat (COCO id) erkannt wird.

In [9]:
# Frames sampeln
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
sample_every = int(max(1, round(fps * 0.25)))  # alle 0.25 s
frame_id = 0
frame_times = []
paths = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_id % sample_every == 0:
        t = frame_id / fps
        p = FRAMES_DIR / f"f_{frame_id:08d}.jpg"
        cv2.imwrite(str(p), frame)
        frame_times.append(t); paths.append(p)
    frame_id += 1

cap.release()
len(paths), "Frames extrahiert"

(4464, 'Frames extrahiert')

In [10]:
model = YOLO("yolov8n.pt")  # klein & schnell

cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
sample_every = int(max(1, round(fps * 0.25)))  # alle 0.25 s
BATCH = 64  # konservativ halten, je nach RAM/GPU

def bgr2rgb(img):
    return img[:, :, ::-1]

times_batch, frames_batch = [], []
det_rows = []

def flush_batch(times_batch, frames_batch):
    if not frames_batch:
        return
    # YOLO direkt auf Arrays → kein PIL/Dateihandle
    results = model(frames_batch, verbose=False)
    for t, r in zip(times_batch, results):
        found = False; max_conf = 0.0
        names = r.names
        for b in r.boxes:
            cls_idx = int(b.cls.item())
            conf = float(b.conf.item())
            if names[cls_idx] == "cat":
                found = True
                max_conf = max(max_conf, conf)
        det_rows.append({"t": t, "cat_present": int(found), "cat_conf": max_conf})
    times_batch.clear(); frames_batch.clear()

frame_id = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break
    if frame_id % sample_every == 0:
        t = frame_id / fps
        times_batch.append(t)
        frames_batch.append(bgr2rgb(frame_bgr))  # YOLO erwartet RGB
        if len(frames_batch) >= BATCH:
            flush_batch(times_batch, frames_batch)
    frame_id += 1

cap.release()
flush_batch(times_batch, frames_batch)  # Reste

cat_df = pd.DataFrame(det_rows).sort_values("t").reset_index(drop=True)
print("Detektionspunkte:", len(cat_df))
cat_df.head()

Detektionspunkte: 4464


,t,cat_present,cat_conf
0,0.000000,0,0.000000
1,0.266667,0,0.000000
2,0.533333,1,0.382030
3,0.800000,1,0.828108
4,1.066667,1,0.372558


# Katzen-Intervalle aus Punkt-Detektionen bilden

Kleine Lücken schließen und zu Intervallen verschmelzen.

In [11]:
# Parameter: max Lücke, die noch als zusammenhängend gilt
gap_close = 0.6  # Sekunden
frame_span = 0.25  # entspricht Sampling-Abstand (oben)

intervals_cat = []
run = None

for _, row in cat_df.iterrows():
    t = float(row.t)
    if row.cat_present:
        if run is None:
            run = {"t_start": t, "t_last": t, "conf_max": float(row.cat_conf)}
        else:
            if t - run["t_last"] <= gap_close:
                run["t_last"] = t
                run["conf_max"] = max(run["conf_max"], float(row.cat_conf))
            else:
                intervals_cat.append({"t_start": run["t_start"], "t_end": run["t_last"]+frame_span, "cat_conf": run["conf_max"]})
                run = {"t_start": t, "t_last": t, "conf_max": float(row.cat_conf)}
# flush
if run is not None:
    intervals_cat.append({"t_start": run["t_start"], "t_end": run["t_last"]+frame_span, "cat_conf": run["conf_max"]})

cat_iv = pd.DataFrame(intervals_cat)
print("Katze-Intervalle:", len(cat_iv))
cat_iv.head(10)

Katze-Intervalle: 235


,t_start,t_end,cat_conf
0,0.533333,1.850000,0.828108
1,4.000000,4.516667,0.520527
2,5.333333,5.583333,0.325119
3,8.000000,10.116667,0.855577
4,13.066667,13.316667,0.452201
5,14.933333,16.516667,0.866572
6,18.400000,18.650000,0.479580
7,22.666667,22.916667,0.314073
8,25.066667,25.316667,0.328689
9,30.133333,30.383333,0.278094


# Schnittmenge (Katze sichtbar ∩ Audio-Event)

In [12]:
def intersect(a0, a1, b0, b1):
    s, e = max(a0, b0), min(a1, b1)
    return (s, e) if e > s else None

out = []
for _, ca in cat_iv.iterrows():
    for _, au in audio_df.iterrows():
        inter = intersect(float(ca.t_start), float(ca.t_end), float(au.t_start), float(au.t_end))
        if inter:
            dur = inter[1] - inter[0]
            if 0.3 <= dur <= 5.0:   # sinnvolle Dauerfenster
                out.append({
                    "t_start": inter[0], "t_end": inter[1], "dur": dur,
                    "cat_conf": float(ca.cat_conf),
                    "audio_score": float(au.audio_score),
                    "centroid": float(au.centroid)
                })

seg_df = pd.DataFrame(out).sort_values(["t_start","dur"]).reset_index(drop=True)
seg_df.to_csv(SEGMENTS_CSV, index=False)
print("Kandidaten-Segmente:", len(seg_df), "→", SEGMENTS_CSV)
seg_df.head(15)

Kandidaten-Segmente: 150 → work_cat_poc/segments.csv


,t_start,t_end,dur,cat_conf,audio_score,centroid
0,1.248000,1.850000,0.602000,0.828108,0.038460,2103.753811
1,4.000000,4.516667,0.516667,0.520527,0.071013,1639.279980
2,8.256000,8.864000,0.608000,0.855577,0.019622,862.397881
3,9.536000,10.116667,0.580667,0.855577,0.029197,1098.435116
4,14.933333,16.516667,1.583333,0.866572,0.034346,1128.020165
5,96.896000,97.760000,0.864000,0.870087,0.021951,997.002708
6,97.856000,98.784000,0.928000,0.870087,0.024087,1280.985255
7,98.880000,99.716667,0.836667,0.870087,0.023940,1379.400550
8,122.016000,122.916667,0.900667,0.789426,0.076348,1346.234674
9,124.864000,125.824000,0.960000,0.929755,0.060706,3470.647287


# Kandidaten als Miniclips speichern

In [18]:
def cut_clip(src_path: Path, dst_path: Path, t0: float, t1: float):
    (
        ffmpeg
        .input(str(src_path), ss=t0, to=t1)
        .output(str(dst_path), c="copy", loglevel="error")
        .overwrite_output()
        .run()
    )

# z.B. die Top-10 nach simplem Score (Audioenergie * Cat-Confidence)
if not seg_df.empty:
    tmp = seg_df.copy()
    tmp["score"] = tmp["audio_score"] * (0.5 + tmp["cat_conf"])  # simple Kombi
    top = tmp.sort_values("score", ascending=False).head(10)
    written = 0
    for i, row in top.iterrows():
        dst = CLIPS_DIR / f"seg_{i:03d}_{row.t_start:.2f}-{row.t_end:.2f}.mp4"
        try:
            cut_clip(VIDEO_PATH, dst, float(row.t_start), float(row.t_end))
            written += 1
        except Exception as e:
            print("Clip-Fehler:", e)
    print(f"{written} Clips in {CLIPS_DIR} geschrieben")
else:
    print("Keine Segmente gefunden – Schwellenwerte anpassen (top_db/centroid/gap_close).")

10 Clips in work_cat_poc/clips geschrieben
